In [1]:
# 필요한 라이브러리 전체 모음 
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field 

import chromadb 
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path

import ast
import json
from pprint import pprint

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


/Users/nanahyun/Documents/GitHub/6th-Trace-ai/trace/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 환경설정
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get('GOOGLE_API_KEY'):
    raise ValueError('GOOGLE_API_KEY가 업서용~ .env 파일을 확인하세요')

print("GOOGLE_API_KEY 설정 완료!")


GOOGLE_API_KEY 설정 완료!


# pydantic 스키마



In [22]:
# 선수개념 
class Prerequisite(BaseModel):
    name: str 
    chapter: str | None = None
    depth: int = 1

# 퀴즈 정보
class QuizItem(BaseModel):
    quiz_id: str         
    question: str
    answer: str          
    concept: str        

class GradeResult(BaseModel):
    quiz_id: str
    is_correct: bool
    feedback: str

# 상태 정의 
# 모든 노드가 공유 
class TraceState(BaseModel):
    question: str # 사용자 질문 
    core_concepts: list[str] = [] # 질문에서 추출한 개념
    prerequisites: list[Prerequisite] = [] # 각 개념에 대한 선수개념
    selected: list[str] = [] # 선택된 개념 
    explanation: str = "" # selected된 개념에 대한 llm의 설명
    quiz: list[QuizItem] = [] # 개념 이해도 퀴즈
    user_answers: dict[str, str] = {}    # {quiz_id: 학생 답}
    grade_results: list[GradeResult] = [] 

만들어야 하는 노드/모듈

1. 사용자 질문에서 수학 개념 추출 

2. 해당 수학 개념으로 지식 그래프 검색해서 선수 개념 찾아오기 -> 가영님 부분 

3. 찾아온 선수개념을 llm이 설명해줌 

4. 벡터 DB에서 그 개념에 해당하는 퀴즈 검색 -> 윤재님

5. 퀴즈 출제


**오케스트레이션 + 인터페이스 설계**

- 체인 조립, FastAPI 엔드포인트 3개, 세션 관리
- 모듈 간 주고받을 딕셔너리 스키마 확정 (Pydantic)
- LLM 출력 파싱 불안정성 대응 

### 노드 1. extracter_node

사용자 질문에서 수학 개념을 추출하는 노드입니다. 

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


# concepts pydantic 모델 정의
class CoreConcept(BaseModel):
    '''사용자 질문에서 핵심 개념 추출을 위한 pydantic 모델'''
    
    core_concepts: list[str] = Field(description="사용자의 질문에서 핵심이 되는 수학 개념들을 추출해라")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
structured_answer = llm.with_structured_output(CoreConcept) # CoreConcept 모델에서 정의한 포맷으로 llm 응답 구조화


#프롬프트 템플릿 정의
extract_prompt = ChatPromptTemplate.from_messages([
    ('system',''' 사용자의 질문에서 핵심이 되는 수학 개념 추출해라.   (지식 그래프 개념 노드를 프롬포트에 넣을 예정)  
'''),
('human','''질문:{question}
 
 이 질문에서 핵심이 되는 수학 개념들을 추출해주세요. 
 
 [knowledge graph의 concept 노드가 프롬포트에 들어갈 예정이다. 총 1631개]
 
 ''')
])




# 각 컴포넌트를 | (파이프) 연산자로 연결해 체인을 만든다.
# 프롬포트와 llm을 연결한 chain 정의 
extracter  = extract_prompt | structured_answer

In [24]:
question = "이차방정식의 근의 공식을 잘 모르겠어요"
result = extracter.invoke({"question": question}) 
print(result)


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 47.737268742s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
# 개념 추출 노드
def extracter_node(state:TraceState) -> dict:
    
    print("\n   [extracter_node] 질문에서 핵심 개념 추출 중...")
    question = state['question'] 
    core_concepts = extract_chain.invoke({"question": question})
    
    return {
        'core_concepts': core_concepts,
        'question' : question
    }

### 노드 2. knowledge_search_node 

지식 그래프에서 선수 개념을 검색하는 노드입니다. 

In [ ]:
# 지식그래프 검색 노드
def knowledge_search_node(state:TraceState) -> dict:
    
    print("\n   [knowledge_search_node] 선수 개념 검색중...")
    question = state["question"]
    core_concepts = state["core_concepts"]


    # 지식그래프 검색 
    # 코드 받아서 구현 필요
    
    
    return {
        'core_concepts': core_concepts,
        'question' : question,
        'prerequisites': []  # 선수개념 검색 결과
    }

In [ ]:
from langchain_core.runnables import RunnableLambda

class Prerequisite(BaseModel):
    """ 선수개념 모델 정의"""
    name: str  # 선수개념 이름
    chapter: str | None = None # 선수개념이 속한 단원 
    depth: int = 1 # 기본 깊이 = 1, 조정 가능함  
    

def mock_graph_query(core_concept: CoreConcept) -> list[Prerequisite]:
    
    """[MOCK] 
    지식그래프 검색 모듈.
    실제로는 core_concepts의 각 개념을 지식그래프에서 찾아서 선수개념을 리턴한다. 
    지금은 체인 조립 확인용 더미 데이터."""
    
    return [
        Prerequisite(name=f"{concept}의 선수개념 OOO (mock)", chapter=None, depth=1)
        for concept in core_concept.core_concepts
    ]

In [ ]:
# 체인 A: 질문 -> 핵심개념 추출 -> 선수개념 검색
chain_a = extracter | RunnableLambda(mock_graph_query) 

result_a = chain_a.invoke({"question": "이차방정식의 근의 공식을 잘 모르겠어요"})
result_a 

[Prerequisite(name='이차방정식의 선수개념 OOO (mock)', chapter=None, depth=1),
 Prerequisite(name='근의 공식의 선수개념 OOO (mock)', chapter=None, depth=1)]

### 노드 3. explainer

사용자가 선택한 수학 개념을 과외 선생님처럼 쉽게 설명하는 llm 응답을 출력하는 노드입니다. 


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# 개념 설명 프롬포트 
# 과외 선생님 페르소나 주입
explain_prompt = ChatPromptTemplate.from_messages([
    ('system', 
    """ 
    너는 수학 과외 선생님이며, 너의 학생은 선행 개념을 모르는 수학 초보자이다. 
    학생에게 수학 개념을 직관적이고 이해하기 쉽게 설명해주어야 한다.
"""),
    ('human', '다음 개념들을 설명해줘: {selected}')
]) 


# 프롬포트, llm, output parser - 세가지 컴포넌트를 연결한 체인 정의
explainer = explain_prompt | llm | StrOutputParser()


In [ ]:
# 테스트
print(explainer.invoke({"selected": ["이차방정식", "근의 공식"]}))

안녕하세요! 수학 과외 선생님입니다. 우리 학생이 수학 초보자라고 하니, 제가 아주 쉽고 직관적으로 설명해 드릴게요. 너무 걱정하지 마세요. 차근차근 따라오면 분명히 이해할 수 있을 거예요!

오늘은 **'이차방정식'**과 **'근의 공식'**이라는 두 가지 중요한 개념을 배워볼 거예요. 이름만 들으면 어려워 보이지만, 사실은 우리 주변에서 많이 쓰이는 아주 유용한 도구들이랍니다.

---

### 1. 이차방정식 (Quadratic Equation)

자, 먼저 **이차방정식**부터 알아볼까요?

**'이차'**라는 말은 '두 번째 차수'라는 뜻이에요. 여기서 '차수'는 미지수(보통 'x'라고 쓰는 알 수 없는 숫자)가 몇 번 곱해졌는지를 나타내는 말이에요.

*   **일차 (1차):** $x$ (x가 한 번 곱해진 것과 같아요. $x^1$이라고도 하죠.)
*   **이차 (2차):** $x^2$ (x가 두 번 곱해진 것, 즉 $x \times x$를 말해요.)
*   **삼차 (3차):** $x^3$ (x가 세 번 곱해진 것, 즉 $x \times x \times x$를 말해요.)

그리고 **'방정식'**은 '등호(=)'를 포함하고 있어서, 미지수(x)의 값을 찾아서 양쪽이 같아지게 만드는 식을 말해요. 마치 양팔 저울처럼 양쪽이 균형을 이루도록 하는 x 값을 찾는 거죠.

**이차방정식은 한마디로,**
**"미지수(x)가 두 번 곱해진($x^2$) 항이 가장 높은 차수로 있는 방정식"**이에요.

가장 기본적인 형태는 이렇게 생겼어요:
$ax^2 + bx + c = 0$

여기서 $a, b, c$는 그냥 어떤 숫자들을 나타내요.
*   $a$는 $x^2$ 앞에 붙은 숫자
*   $b$는 $x$ 앞에 붙은 숫자
*   $c$는 혼자 있는 숫자 (상수항이라고 불러요)

**중요한 점은!** $a$는 절대로 0이 되면 안 돼요. 만약 $a$가 0이 되면 $x^2$ 항이 사라져서 이차방정식이 아니라 일차방정식이 되어버리거든요.

**예시를 들어볼까요?**
*

### 노드 4. 퀴즈 출제 

설명한 개념을 이해했는지 확인하기 위해 퀴즈를 출제합니다.  벡터DB에 저장된 수학 문제를 사용합니다. 

In [ ]:
import chromadb 
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path

# 프로젝트 루트
PROJECT_ROOT = Path().resolve().parent  # 현재 폴더에서 한 단계 위 폴더 
# 프로젝트 루트 기준으로 경로를 지정
DB_PATH = str(PROJECT_ROOT / "data" / "chromadb")

# 임베딩 모델 로드
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")


# 벡터 DB 로드
client = chromadb.PersistentClient(path=DB_PATH)

vectorstore = Chroma(
    client=client,
    collection_name="problem_collection",
    embedding_function=embeddings,
)

# 예시 문서 1개 조회
docs = vectorstore.similarity_search("이차방정식", k=1)
sample_doc = docs[0]
print(sample_doc.page_content)
print(sample_doc.metadata)

In [ ]:
def quiz_retriever(selected: list[str]) -> list[QuizItem]:
    """벡터DB 퀴즈 검색 모듈.
    개념별로 vectorstore.similarity_search(concept, k=5)로 문제 5개(조정 가능)씩 가져온다.
    """
    
    # 현재 코드는 임시
    # 코드 받아서 구현 필요
    
    quizzes = []
    for concept in selected:
        docs = vectorstore.similarity_search(concept, k=5)
        for i, doc in enumerate(docs):
            quizzes.append(
                QuizItem(
                    quiz_id=f"{concept}-{i}",
                    question=doc.page_content,
                    answer=doc.page_content,
                    concept=concept,
                )
            )
    return quizzes


quiz_prompt = ChatPromptTemplate.from_messages([
    ('system', 
    """ 
    너는 수학 과외 선생님이며, 너의 학생은 선행 개념을 모르는 수학 초보자이다. 
    지금은 개념을 설명해준 뒤이다. 이제부터는 해당 개념을 이해했는지 확인하기 위해 퀴즈를 내야한다.
    제공하는 퀴즈를 보고, 학생에게 내기 좋은 형태로 퀴즈를 출제해라.
    문제들을 먼저 전부 보여주고, 학생이 다 풀어볼 수 있도록 한 뒤,
    마지막에 "정답 확인" 섹션을 따로 만들어서 문제 번호별 정답을 같이 제시해라.
    (학생이 다 풀고 나서 스스로 정답을 맞춰볼 수 있도록)
"""),
    ('human', '{quizzes}\n\n위 문제들을 참고해서 다음 개념들에 대한 퀴즈를 출제해줘: {selected}')
])

# 체인 B: 선택된 개념 -> 퀴즈 검색 -> llm이 출제 형태로 가공
chain_b = (
    RunnableLambda(lambda x: {"quizzes": quiz_retriever(x["selected"]), "selected": x["selected"]})
    | quiz_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
result_b = chain_b.invoke({"selected": ["이차방정식"]})
print(result_b)

안녕하세요! 이차방정식 개념 설명은 잘 들었죠? 이제 배운 내용을 얼마나 이해했는지 확인해 볼 시간이에요. 너무 부담 갖지 말고, 아는 만큼 최선을 다해서 풀어보세요. 모르는 부분이 있다면 다시 개념을 찾아봐도 좋고, 나중에 저에게 질문해도 괜찮아요!

---

**[이차방정식 개념 확인 퀴즈]**

**문제 1.** 다음 중 이차방정식인 것을 모두 고르세요.
(1) $x + 5 = 0$
(2) $x^2 - 3x + 1 = 0$
(3) $2x^3 + x^2 - 7 = 0$
(4) $4x^2 - 9 = 0$
(5) $x^2 = x^2 + 2x - 3$

<br/>

**문제 2.** 이차방정식 $5x^2 - 2x + 7 = 0$ 에서 다음을 각각 말해보세요.
(1) $x^2$의 계수
(2) $x$의 계수
(3) 상수항

<br/>

**문제 3.** 이차방정식 $x^2 - 5x + 6 = 0$ 에 대해 다음 물음에 답하세요.
(1) $x=1$ 이 이 이차방정식의 해가 되는지 확인해 보세요.
(2) $x=2$ 가 이 이차방정식의 해가 되는지 확인해 보세요.

<br/>

**문제 4.** 다음 이차방정식의 해를 구해보세요.
(1) $x^2 = 16$
(2) $(x-3)^2 = 0$

<br/>

**문제 5.** 이차방정식 $x^2 + 2x - 8 = 0$ 의 해를 구해보세요. (힌트: 인수분해를 이용해 보세요!)

---

다 풀어보셨나요? 이제 아래에서 정답을 확인해 보세요!

---

**[정답 확인]**

**문제 1.**
**정답:** (2), (4)
**설명:**
*   (1) $x$의 최고차항이 1차이므로 일차방정식입니다.
*   (2) $x$의 최고차항이 2차이고, $x^2$의 계수가 0이 아니므로 이차방정식입니다.
*   (3) $x$의 최고차항이 3차이므로 삼차방정식입니다.
*   (4) $x$의 최고차항이 2차이고, $x^2$의 계수가 0이 아니므로 이차방정식입니다.
*   (5) $x^2$ 항을 이항하면 $x^2 - x^2 - 2x + 3 =

### 참고 - 벡터DB document 형식

In [25]:
# meta data 확인
from pprint import pprint
import json

pprint(sample_doc.metadata, width=100, sort_dicts=False)


{'source_data_info.types_of_problems': '객관식',
 'source_file': '/content/extracted_training/S3_중등_2_016024.json',
 'source_data_info.2009_achievement_standard': "[' ']",
 'raw_data_info.subject': '수학',
 'raw_data_info.grade': '2학년',
 'source_data_info.level_of_difficulty': '하',
 'source_data_info.2022_achievement_standard': "['[9수02-13] 미지수가 2개인 연립일차방정식을 풀 수 있고, 이를 활용하여 문제를 "
                                               "해결할 수 있다.']",
 'raw_data_info.school': '중학교',
 'source_data_info.2015_achievement_standard': "['[9수02-11] 미지수가 2개인 연립일차방정식을 풀 수 있고, 이를 활용하여 문제를 "
                                               "해결할 수 있다.']",
 'raw_data_info.semester': '1학기'}


In [26]:
import ast
from pprint import pprint

# page content 확인
parsed = ast.literal_eval(sample_doc.page_content)  # 문자열 → 실제 list/dict로 복원
pprint(parsed, width=100, sort_dicts=False)


[{'class_num': 1,
  'class_name': '문항(텍스트)',
  'class_info_list': [{'Type': 'Bounding_Box',
                       'Type_value': [[12.46, 37.91, 245.71, 63.66]],
                       'text_description': '다음 연립방정식을 풀어라.'}]},
 {'class_num': 1,
  'class_name': '문항(이미지)',
  'class_info_list': [{'Type': 'Bounding_Box',
                       'Type_value': [[374.96, 81.58, 505.71, 159.33]],
                       'text_description': '$\\left\\{\\begin{array}{l}x-y=2 \\\\ 2 '
                                           'x+y=7\\end{array}\\right. $'}]},
 {'class_num': 1,
  'class_name': '정답(텍스트)',
  'class_info_list': [{'Type': 'Bounding_Box',
                       'Type_value': [[11.96, 179.83, 164.21, 206.83]],
                       'text_description': '① $x=3, y=1$'}]},
 {'class_num': '1',
  'class_name': '정답(이미지)',
  'class_info_list': [{'Type': 'Bounding_Box',
                       'Type_value': [[68.21, 739.33, 218.21, 766.83]],
                       'text_description': '① $x=3, y=1